# 02b · Reason codes (step 8)

**Owner:** Joseph (took over step 8 from Germain). **Model:** the step 7 choice, logistic regression + Platt ([ADR 0016](../docs/adr/0016-final-classification.md)). **Decision record:** [ADR 0017](../docs/adr/0017-reason-codes.md). **Plain write-up:** [learnings/08](../learnings/08_reason_codes.md).

When NkwaPay declines someone it must say why. This notebook tests every way we could produce the **top 3 reasons**, using `src/reasons.py` (the code the scorer will call).

**Privacy:** every output here is a count or a share. No applicant's row or sentence is printed; example sentences use made-up applicants.

In [1]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, shap
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import log_loss, average_precision_score
from src.features import build, FEATURES, preprocessor, splits
from src.classification_experiments import final_model, logit
from src.reasons import reference, pushes, reasons, THEMES, clue_of, MIN_PUSH
TABLES = ROOT / "learnings" / "tables"
tr, te = build(str(ROOT / "data" / "zindi"))
print(len(tr), "labelled loans;", len(te), "unlabelled applicants (Zindi test)")

68464 labelled loans; 18594 unlabelled applicants (Zindi test)


## 0. First, a bug in the saved step 7 model

Step 7's final model was calibrated on the newest 20% of *dates*. On the full data that's May–Nov 2024 only: 2,625 loans with a 9% default rate. The model then priced every loan like a 2024 loan, so it approved only 12% of 2022 loans and 11% of the Zindi test applicants.

We test five ways to calibrate the final model on every future-loan block. (Step 7's own tests calibrated each block separately, which is why they didn't show this.)

In [2]:
lr = lambda: make_pipeline(preprocessor(FEATURES), LogisticRegression(max_iter=2000))
def platt(base, Xc, yc):
    c = LogisticRegression().fit(logit(base.predict_proba(Xc)[:, 1]).reshape(-1, 1), yc)
    return lambda X: c.predict_proba(logit(base.predict_proba(X)[:, 1]).reshape(-1, 1))[:, 1]
def options(L):
    out = {"none (plain LR)": lambda X, m=lr().fit(L[FEATURES], L.target): m.predict_proba(X)[:, 1]}
    o = L.sort_values("disbursement_date", kind="mergesort")
    for name, b in [("Platt, newest 20% of dates (step 7)", pd.Timestamp(np.sort(o.disbursement_date.unique())[int(.8 * (o.disbursement_date.nunique() - 1))])),
                    ("Platt, newest 20% of rows", o.disbursement_date.iloc[int(.8 * len(o))])]:
        fit, cal = o[(o.disbursement_date < b) & (o.outcome_known_at < b)], o[o.disbursement_date >= b]
        out[name] = platt(lr().fit(fit[FEATURES], fit.target), cal[FEATURES], cal.target)
    a, b = train_test_split(L.index, test_size=.25, random_state=0, stratify=L.target)
    out["Platt, random 25%"] = platt(lr().fit(L.loc[a, FEATURES], L.loc[a, "target"]), L.loc[b, FEATURES], L.loc[b, "target"])
    out["CHOSEN: Platt on out-of-fold scores, 5 customer folds"] = lambda X, m=final_model(L): m.predict_proba(X)[:, 1]
    return out
rows = []
for f, (learn, test) in enumerate(splits(tr, "time_block")):
    if not len(learn): continue
    L, T = tr.loc[learn], tr.loc[test]; y = T.target.to_numpy()
    for name, pred in options(L).items():
        p = pred(T[FEATURES]); a = p <= .021
        rows.append(dict(block=f, option=name, n=len(y), actual=y.mean(), predicted=p.mean(), log_loss=log_loss(y, p),
                         lift=average_precision_score(y, p) / y.mean(), approved=a.mean(),
                         cost_KES=((a & (y == 1)).sum() * 4700 + (~a & (y == 0)).sum() * 100) / len(y)))
cal = pd.DataFrame(rows); cal.to_csv(TABLES / "08_final_calibration.csv", index=False)
w = cal.assign(**{c: cal[c] * cal.n for c in ["predicted", "actual", "log_loss", "approved", "cost_KES"]}).groupby("option")[["predicted", "actual", "log_loss", "approved", "cost_KES", "n"]].sum()
w.drop(columns="n").div(w.n, axis=0).round(4).sort_values("cost_KES")

,predicted,actual,log_loss,approved,cost_KES
option,,,,,
"Platt, random 25%",0.0139,0.0181,0.0785,0.8874,47.2559
"Platt, newest 20% of rows",0.0149,0.0181,0.0791,0.8702,47.8343
none (plain LR),0.0135,0.0181,0.0801,0.8935,47.8747
"CHOSEN: Platt on out-of-fold scores, 5 customer folds",0.0134,0.0181,0.0799,0.8930,47.9298
"Platt, newest 20% of dates (step 7)",0.0146,0.0181,0.0949,0.8930,48.5449


In [3]:
m = final_model(tr)   # the fixed final model (also what src/classification_experiments.py now saves)
p_tr = m.predict_proba(tr[FEATURES])[:, 1]
print("Platt slope %.3f (1 = Platt changes nothing)" % m.calibrator.coef_[0, 0])
display(pd.DataFrame({"year": tr.disbursement_date.dt.year, "p": p_tr, "y": tr.target}).groupby("year").agg(
    loans=("y", "size"), predicted=("p", "mean"), actual=("y", "mean"), approved_at_2_1=("p", lambda s: (s <= .021).mean())).round(3))
p_te = m.predict_proba(te[FEATURES])[:, 1]
cut = np.where(te.country_id == "Ghana", .032, .021)
print(te.assign(p=p_te, declined=p_te > cut).groupby("country_id").agg(applicants=("p", "size"), mean_risk=("p", "mean"), declined=("declined", "mean")).round(3))

Platt slope 0.989 (1 = Platt changes nothing)


,loans,predicted,actual,approved_at_2_1
year,,,,
2021,39,0.511,0.538,0.000
2022,64402,0.013,0.013,0.853
2023,1239,0.120,0.115,0.052
2024,2784,0.095,0.093,0.001


            applicants  mean_risk  declined
country_id                                 
Ghana             3525      0.125     0.474
Kenya            15069      0.018     0.192


## 1. Who we explain, and against whom

We explain the **declined** Zindi test applicants (2.1% cut-off in Kenya, 3.2% in Ghana). Each reason is a *push*: how much one theme raises this person's risk above a **reference applicant**.

In [4]:
dec = te[p_te > cut]; X = dec[FEATURES]
groups = {"Kenya": dec.country_id == "Kenya", "Ghana": dec.country_id == "Ghana", "first-timers (no earlier loans)": dec.n_prev_loans == 0}
known = set(tr.loan_type)
ref = reference(m, tr[FEATURES])
top1 = lambda P: P.idxmax(axis=1)
def topk(P, k=3, floor=MIN_PUSH):
    return [frozenset(r[r >= floor].sort_values(ascending=False).index[:k]) for _, r in P.iterrows()]
same = lambda A, B: float(np.mean([a == b for a, b in zip(A, B)]))
print({k: int(v.sum()) for k, v in groups.items()}, "declined applicants")

{'Kenya': 2894, 'Ghana': 1672, 'first-timers (no earlier loans)': 2033} declined applicants


## 2. Test: how to compute a push

For a logistic regression, *weight × (value − reference)* is exactly what SHAP computes in its standard form, so we check that they match. SHAP also has a **correlation-aware** form that shares credit between clues that move together; that one can differ.

In [5]:
pre, lrm, slope = m.base[0], m.base[-1], m.calibrator.coef_[0, 0]
bg = tr[FEATURES].sample(2000, random_state=0); Zbg, Zx = pre.transform(bg), pre.transform(X)
ours = pushes(m, X, Zbg.mean(0), "column")
sv = shap.LinearExplainer(lrm, shap.maskers.Independent(Zbg, max_samples=2000)).shap_values(Zx) * slope
svc = shap.LinearExplainer(lrm, shap.maskers.Impute(Zbg)).shap_values(Zx) * slope
to_theme = lambda A: pd.DataFrame(A, columns=ours.columns, index=X.index).T.groupby(ours.columns.map(lambda c: THEMES[clue_of(c)])).sum().T
T_ours, T_corr = to_theme(ours.to_numpy()), to_theme(svc)
t1 = pd.DataFrame([dict(comparison="ours vs SHAP standard", largest_difference=float(np.abs(ours.to_numpy() - sv).max()),
                        same_top_reason=float((top1(to_theme(sv)) == top1(T_ours)).mean()), same_top3=same(topk(to_theme(sv)), topk(T_ours))),
                   dict(comparison="ours vs SHAP correlation-aware", largest_difference=float(np.abs(ours.to_numpy() - svc).max()),
                        same_top_reason=float((top1(T_corr) == top1(T_ours)).mean()), same_top3=same(topk(T_corr), topk(T_ours)))])
t1.to_csv(TABLES / "08_method.csv", index=False); display(t1.round(4))
pd.DataFrame({"ours": top1(T_ours).value_counts(normalize=True), "SHAP correlation-aware": top1(T_corr).value_counts(normalize=True)}).round(3)

Estimating transforms:   0%|          | 0/1000 [00:00<?, ?it/s]

Estimating transforms:  20%|██        | 205/1000 [00:00<00:00, 2046.26it/s]

Estimating transforms:  41%|████▏     | 413/1000 [00:00<00:00, 2062.93it/s]

Estimating transforms:  63%|██████▎   | 626/1000 [00:00<00:00, 2090.62it/s]

Estimating transforms:  84%|████████▍ | 839/1000 [00:00<00:00, 2106.01it/s]

Estimating transforms: 100%|██████████| 1000/1000 [00:00<00:00, 2093.79it/s]

,comparison,largest_difference,same_top_reason,same_top3
0,ours vs SHAP standard,0.0000,1.0000,1.0000
1,ours vs SHAP correlation-aware,2.7087,0.7258,0.8533


,ours,SHAP correlation-aware
borrowing history,0.700,0.475
loan length,0.159,0.130
loan product,0.141,0.395
loan sizes,NaN,0.000
repayment record,0.000,NaN


## 3. Test: who is the reference applicant?

* **average**: the average loan in training.
* **approved**: the average loan the model approves.
* **best**: every clue at its least risky value (the "maximum points" method some credit bureaus use).

In [6]:
refs = {"average": ref, "approved": reference(m, tr[FEATURES], "approved", .021), "best": reference(m, tr[FEATURES], "best")}
T = {k: pushes(m, X, r) for k, r in refs.items()}
rows = []
for k, t in T.items():
    s = top1(t).value_counts(normalize=True)
    rows.append(dict(reference=k, same_top_reason_as_average=float((top1(t) == top1(T["average"])).mean()),
                     same_top3_as_average=same(topk(t), topk(T["average"])), mean_reasons=float((t >= MIN_PUSH).sum(axis=1).clip(upper=3).mean()),
                     most_common_top_reason=s.index[0], its_share=float(s.iloc[0])))
t2 = pd.DataFrame(rows); t2.to_csv(TABLES / "08_reference.csv", index=False); t2.round(3)

,reference,same_top_reason_as_average,same_top3_as_average,mean_reasons,most_common_top_reason,its_share
0,average,1.000,1.000,1.867,borrowing history,0.699
1,approved,0.968,0.987,1.877,borrowing history,0.732
2,best,0.141,0.115,3.000,loan product,1.000


## 4. Test: how to group, and how strong a reason must be

Clues that move together can cancel out. Example: a large *last* loan pushes risk up, and a large *usual* loan pushes it down, and the two are 0.89 correlated. Shown separately, one of them becomes a "reason" that its partner mostly undoes. We count how often a listed reason is more than half cancelled by the other clues in its theme, and how often the same applicant gets the same top 3 when the model is refitted on other data (stability, next section).

In [7]:
v1 = dict(THEMES, n_finished="repayment record")                       # first draft: finished loans with repayment
levels = {"13 clues": ("clue", None), "themes, draft 1": ("theme", v1), "themes, final": ("theme", THEMES)}
C_ = pushes(m, X, ref, "clue")
def cancelled(level, themes):
    if level == "clue":   # a clue reason is 'cancelled' if its final theme's total is < half of it
        th = pushes(m, X, ref, "theme")
        return float(np.mean([th.loc[i, THEMES[c]] < .5 * C_.loc[i, c] for i in X.index for c in topk(C_.loc[[i]])[0]]))
    if themes is v1:      # draft 1: finished loans in another theme than past loans
        th1, th2 = pushes(m, X, ref, "theme", v1), pushes(m, X, ref, "theme")
        return float(np.mean([th2.loc[i, "borrowing history"] < .5 * th1.loc[i, t] for i in X.index for t in topk(th1.loc[[i]])[0] if t in ("borrowing history", "repayment record")] or [0]))
    return 0.0            # final themes are the unit, nothing outside cancels them
rows = [dict(grouping=k, cancelled_share=cancelled(*v), distinct_top_reasons=int(pushes(m, X, ref, *[v[0]] + ([v[1]] if v[1] else [])).idxmax(axis=1).nunique())) for k, v in levels.items()]
t3 = pd.DataFrame(rows); t3.to_csv(TABLES / "08_grouping.csv", index=False); t3

,grouping,cancelled_share,distinct_top_reasons
0,13 clues,0.086847,7
1,"themes, draft 1",0.000222,4
2,"themes, final",0.000000,4


In [8]:
# Stability: refit the model on each future-loan block's learning rows and on 10 customer bootstraps;
# does the same applicant keep the same reasons? Tested for each grouping and minimum push.
refits = []
for f, (learn, _) in enumerate(splits(tr, "time_block")):
    if len(learn): refits.append((f"time block {f}", final_model(tr.loc[learn])))
rng = np.random.default_rng(0); cust = tr.customer_id.unique()
for b in range(10):
    pick = pd.Series(rng.choice(cust, len(cust))).value_counts()
    s = tr.set_index("customer_id").loc[pick.index.repeat(pick.values)].rename_axis("customer_id").reset_index()
    refits.append((f"bootstrap {b}", final_model(s)))
rows = []
for gname, (level, themes) in levels.items():
    kw = {"themes": themes} if themes else {}
    main = pushes(m, X, ref, level, **kw)
    for floor in (0.0, 0.05, 0.1, 0.2):
        base = topk(main, floor=floor)
        for rname, r in refits:
            P = pushes(r, X, reference(r, tr[FEATURES]), level, **kw)
            rows.append(dict(grouping=gname, min_push=floor, refit=rname, same_top_reason=float((top1(P) == top1(main)).mean()),
                             same_top3=same(topk(P, floor=floor), base), mean_reasons=float(np.mean([len(x) for x in base])),
                             no_reason=float(np.mean([len(x) == 0 for x in base]))))
st = pd.DataFrame(rows); st.to_csv(TABLES / "08_stability.csv", index=False)
st.groupby(["grouping", "min_push"])[["same_top_reason", "same_top3", "mean_reasons", "no_reason"]].agg(["mean", "min"]).round(3)

same_top_reason        same_top3        mean_reasons  \
                                    mean    min      mean    min         mean   
grouping        min_push                                                        
13 clues        0.00               0.766  0.573     0.547  0.126        3.000   
                0.05               0.766  0.573     0.532  0.130        2.990   
                0.10               0.766  0.573     0.533  0.128        2.980   
                0.20               0.766  0.573     0.529  0.121        2.956   
themes, draft 1 0.00               0.907  0.805     0.574  0.271        2.325   
                0.05               0.907  0.805     0.587  0.222        2.145   
                0.10               0.907  0.805     0.721  0.223        1.979   
                0.20               0.907  0.805     0.862  0.285        1.866   
themes, final   0.00               0.908  0.827     0.625  0.269        2.399   
                0.05               0.908  0.827     0.611  0.277        2.143   
                0.10               0.908  0.827     0.775  0.406        1.980   
                0.20               0.908  0.827     0.938  0.635        1.867   

                                no_reason       
                            min      mean  min  
grouping        min_push                        
13 clues        0.00      3.000       0.0  0.0  
                0.05      2.990       0.0  0.0  
                0.10      2.980       0.0  0.0  
                0.20      2.956       0.0  0.0  
themes, draft 1 0.00      2.325       0.0  0.0  
                0.05      2.145       0.0  0.0  
                0.10      1.979       0.0  0.0  
                0.20      1.866       0.0  0.0  
themes, final   0.00      2.399       0.0  0.0  
                0.05      2.143       0.0  0.0  
                0.10      1.980       0.0  0.0  
                0.20      1.867       0.0  0.0

## 5. Test: who gets which reasons?

The reasons should make sense for every group, especially Ghanaians (no known outcomes; 87% on loan type `Type_3`, never seen in Kenya) and first-timers.

In [9]:
Th = pushes(m, X, ref)
rows = []
for g, mask in groups.items():
    t = Th[mask.to_numpy()]
    s = top1(t).value_counts(normalize=True)
    rows.append(dict(group=g, declined=int(mask.sum()), **{f"top reason: {k}": float(v) for k, v in s.items()},
                     no_reason=float(((t >= MIN_PUSH).sum(axis=1) == 0).mean())))
t5 = pd.DataFrame(rows).fillna(0); t5.to_csv(TABLES / "08_groups.csv", index=False); t5.round(3).T

,0,1,2
group,Kenya,Ghana,first-timers (no earlier loans)
declined,2894,1672,2033
top reason: borrowing history,0.693,0.711,0.777
top reason: loan product,0.223,0.0,0.026
top reason: loan length,0.084,0.289,0.197
top reason: repayment record,0.0,0.0,0.0
no_reason,0.0,0.0,0.0


## 6. Test: the wording

Every sentence is filled in with the applicant's own numbers. We check that every declined applicant gets a sentence without blanks, and count which kinds of sentence come out (counts only).

In [10]:
R = reasons(m, X, ref, known_types=known)
kind = lambda s: (s.split(";")[0].split("(")[0].rstrip(". ").translate(str.maketrans("0123456789", "#" * 10)))
k = pd.Series([kind(s) for x in R for s in x]).str.replace(r"#+(,#+)*", "#", regex=True)
print("reasons per declined applicant:", pd.Series(map(len, R)).value_counts().sort_index().to_dict(), "| any blank 'nan':", any("nan" in s for x in R for s in x))
t6 = k.value_counts().rename("count").to_frame(); t6.to_csv(TABLES / "08_wording.csv"); t6

reasons per declined applicant: {1: 1497, 2: 2180, 3: 889} | any blank 'nan': False


,count
"This is your first loan with us, so we have no borrowing history to go on",2033
You took no loans in the last # days,1577
Loans of this kind,1489
We have no repayment record for this kind of loan,1226
The loan is for # days,1185
"You have only # earlier loan with us, so we know little about how you repay",811
This is a first loan with this lender,68
"You have only # earlier loans with us, so we know little about how you repay",48
Your most recent loan,44
Your last loan was # days ago,18


In [11]:
# Three made-up applicants (not real rows), to show what a borrower would read.
fake = pd.DataFrame([
    dict(loan_type="Type_1", duration=7, New_versus_Repeat="Repeat Loan", n_prev_loans=2, n_finished=1, n_defaults=0, past_default_rate=0.0,
         no_known_history=0, days_since_last_loan=40.0, n_loans_last_7d=0, n_loans_last_30d=0, mean_prev_amount=3000.0, last_amount=9000.0),
    dict(loan_type="Type_3", duration=30, New_versus_Repeat="Repeat Loan", n_prev_loans=5, n_finished=0, n_defaults=0, past_default_rate=np.nan,
         no_known_history=1, days_since_last_loan=9.0, n_loans_last_7d=0, n_loans_last_30d=2, mean_prev_amount=800.0, last_amount=800.0),
    dict(loan_type="Type_1", duration=7, New_versus_Repeat="New Loan", n_prev_loans=0, n_finished=0, n_defaults=0, past_default_rate=np.nan,
         no_known_history=1, days_since_last_loan=np.nan, n_loans_last_7d=0, n_loans_last_30d=0, mean_prev_amount=np.nan, last_amount=np.nan)],
    index=["Kenyan repeat borrower", "Ghanaian, unseen loan type", "first-ever loan"])[FEATURES]
for name, rs, p in zip(fake.index, reasons(m, fake, ref, known_types=known), m.predict_proba(fake)[:, 1]):
    print(f"{name}: risk {p:.1%}"); [print("  -", s) for s in rs]

Kenyan repeat borrower: risk 3.8%
  - You took no loans in the last 7 days; regular borrowers have repaid more reliably.
  - Your most recent loan (9,000) was larger than your usual loan (3,000).
Ghanaian, unseen loan type: risk 13.2%
  - The loan is for 30 days; longer loans are repaid less often.
  - You took no loans in the last 7 days; regular borrowers have repaid more reliably.
  - We have no repayment record for this kind of loan (Type_3), so we treat it with caution.
first-ever loan: risk 4.8%
  - This is your first loan with us, so we have no borrowing history to go on.
  - This is a first loan with this lender; first loans are repaid less often than repeat loans.
